# 6. CHURN ATTRIBUTION AND EXPLAINABILITY
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../models/churn_model_YYYYMMDD.joblib`, `../data/processed/churn_features_YYYYMMDD.parquet`, and `../data/processed/churn_predictions_YYYYMMDD.parquet`

*The trained XGBoost model, the modeling matrix, and the scored customer snapshots from NB04.*

**OUTPUT:** `../data/processed/churn_explainability_YYYYMMDD.parquet`, `../data/processed/churn_driver_summary_YYYYMMDD.csv`, and `../reports/churn_explainability_YYYYMMDD.html`

*A SHAP-based explainability package connecting churn predictions to actionable drivers, customer profiles, and retention decisions.*


---
## 6.1. STARTING SITUATION


NB05 established that the canonical V2C ranking is usable for campaign prioritization. The next business question is not only *who* is high risk, but *why* each customer is high risk and what kind of action should be triggered.

This notebook therefore turns model scores into **interpretable churn drivers**. It uses SHAP to quantify the factors pushing risk upward or downward and maps those drivers into the retention-action framework defined for VivaMarket Brasil.

---
## 6.2. NOTEBOOK OBJECTIVE


- **Business objective:** connect churn predictions to concrete retention actions such as win-back, logistics apology, category reactivation, or VIP escalation.
- **Analytical objective:** compute SHAP-based global and local explanations, summarize dominant drivers by risk tier, and prepare explainability outputs for orchestration and reporting on the canonical V2C line.

In [1]:
import base64
import io
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb06_explainability')
logger.info('NB06 started: churn attribution and explainability.')

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


2026-05-06 08:28:32,258 | INFO | NB06 started: churn attribution and explainability.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

run_date_tag = datetime.now(ZoneInfo('Europe/Paris')).strftime('%Y%m%d')
model_path = sorted(MODELS_DIR.glob('churn_model_*.joblib'))[-1]
feature_path = sorted(PROCESSED_DIR.glob('churn_features_*.parquet'))[-1]
prediction_path = sorted(PROCESSED_DIR.glob('churn_predictions_*.parquet'))[-1]
explainability_path = PROCESSED_DIR / f'churn_explainability_{run_date_tag}.parquet'
driver_summary_path = PROCESSED_DIR / f'churn_driver_summary_{run_date_tag}.csv'
explainability_html_path = REPORTS_DIR / f'churn_explainability_{run_date_tag}.html'


In [3]:
package = joblib.load(model_path)
model = package['model']
feature_columns = package['feature_columns']
feature_df = pd.read_parquet(feature_path)
feature_df['snapshot_date'] = pd.to_datetime(feature_df['snapshot_date'])
prediction_df = pd.read_parquet(prediction_path)
prediction_df['snapshot_date'] = pd.to_datetime(prediction_df['snapshot_date'])
target_column = package.get('target_column', 'churn_v2_label' if 'churn_v2_label' in feature_df.columns else 'churn_90d_label')

leakage_columns = [
    'customer_unique_id', 'snapshot_key', 'snapshot_date', 'first_purchase_timestamp',
    'last_purchase_timestamp', 'future_orders_90d', 'future_revenue_90d', 'churn_90d_label',
    'churn_v2_label', 'next_purchase_timestamp', 'days_to_next_purchase', 'future_purchase_within_horizon'
]
test_df = (
    feature_df[feature_df['snapshot_key'].astype(str).isin([str(k) for k in package['test_snapshot_keys']])]
    .copy()
    .reset_index()
    .rename(columns={'index': 'source_row_id'})
)
X_test = pd.get_dummies(
    test_df[[c for c in feature_df.columns if c not in leakage_columns]],
    columns=['customer_state'],
    dtype=float,
).reindex(columns=feature_columns, fill_value=0.0)
X_test.index = test_df['source_row_id']

test_df['observed_target'] = test_df[target_column].astype(int)

support_columns = [
    'source_row_id', 'customer_unique_id', 'snapshot_key', 'snapshot_date', 'customer_state',
    'avg_review_score', 'late_delivery_rate_total', 'revenue_90d', 'credit_card_share_total',
    'boleto_share_total', 'voucher_share_total', 'total_payment_value', 'observed_target'
]
existing_support_columns = [c for c in support_columns if c in test_df.columns]

explainability_base = prediction_df.merge(
    test_df[existing_support_columns],
    on=['customer_unique_id', 'snapshot_key', 'snapshot_date'],
    how='left',
    validate='one_to_one',
)

for logical_name in ['observed_target', 'total_payment_value']:
    x_col = f'{logical_name}_x'
    y_col = f'{logical_name}_y'
    if x_col in explainability_base.columns and y_col in explainability_base.columns:
        explainability_base[logical_name] = explainability_base[x_col].fillna(explainability_base[y_col])
        explainability_base = explainability_base.drop(columns=[x_col, y_col])
    elif logical_name not in explainability_base.columns:
        explainability_base[logical_name] = np.nan

explainability_base['observed_target'] = explainability_base['observed_target'].astype(int)
explainability_base = explainability_base.set_index('source_row_id')
explainability_base.head()

,customer_unique_id,snapshot_key,snapshot_date,recency_days,total_orders,orders_30d,orders_90d,churn_probability,risk_tier,selected_model,version_name,customer_state,avg_review_score,late_delivery_rate_total,revenue_90d,credit_card_share_total,boleto_share_total,voucher_share_total,observed_target,total_payment_value
source_row_id,,,,,,,,,,,,,,,,,,,,
6225,004288347e5e88a27ded2bb23747066c,20180401,2018-04-01,77,2,0.0000,1.0000,0.9101,LOW,xgboost,v2,RJ,5.0000,0.0000,103.2800,1.0000,0.0000,0.0000,1,354.3700
6226,00cc12a6d8b578b8ebd21ea4e2ae8b27,20180401,2018-04-01,376,2,0.0000,0.0000,0.9545,LOW,xgboost,v2,SP,4.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1,126.2000
6227,011b4adcd54683b480c4d841250a987f,20180401,2018-04-01,45,2,0.0000,1.0000,0.8569,LOW,xgboost,v2,BA,4.5000,0.0000,149.8800,1.0000,0.0000,0.0000,1,236.3000
6228,013f4353d26bb05dc6652f1269458d8d,20180401,2018-04-01,124,2,0.0000,0.0000,0.6468,LOW,xgboost,v2,BA,5.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1,356.3900
6229,015557c9912277312b9073947804a7ba,20180401,2018-04-01,335,2,0.0000,0.0000,0.9673,MEDIUM,xgboost,v2,SP,5.0000,0.0000,0.0000,0.2313,0.7687,0.0000,1,315.1200


In [4]:
sample_n = min(5000, len(X_test))
X_sample = X_test.sample(n=sample_n, random_state=42).sort_index()
base_sample = explainability_base.loc[X_sample.index].copy()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)
shap_array = np.asarray(shap_values)
if shap_array.ndim == 3:
    shap_array = shap_array[..., -1]
logger.info('SHAP matrix shape: %s', shap_array.shape)

mean_abs_shap = np.abs(shap_array).mean(axis=0)
global_importance = (
    pd.DataFrame({'feature': X_sample.columns, 'mean_abs_shap': mean_abs_shap})
    .sort_values('mean_abs_shap', ascending=False)
    .reset_index(drop=True)
)
global_importance.head(15)

2026-05-06 08:28:33,064 | INFO | SHAP matrix shape: (3346, 112)


,feature,mean_abs_shap
0,mean_gap_days,0.6512
1,recency_days,0.5755
2,total_freight_value,0.5514
3,credit_card_value_180d,0.3561
4,median_gap_days,0.3139
5,total_item_price,0.2755
6,total_payment_value,0.2637
7,total_items,0.2382
8,tenure_days,0.2234
9,revenue_180d,0.2036


In [5]:
def map_driver(feature_name: str) -> str:
    name = feature_name.lower()
    if 'recency' in name or 'active_last' in name:
        return 'recency'
    if 'late_delivery' in name or 'review' in name:
        return 'logistics'
    if 'frequency' in name or 'orders_' in name or 'mean_gap' in name:
        return 'frequency'
    if 'revenue' in name or 'payment' in name or 'monetary' in name or 'value' in name:
        return 'monetary'
    if 'category' in name or 'product' in name:
        return 'category'
    return 'other'

top_idx = np.abs(shap_array).argmax(axis=1)
top_features = [X_sample.columns[i] for i in top_idx]
top_feature_shap = shap_array[np.arange(len(X_sample)), top_idx]

explainability_sample = base_sample.copy()
explainability_sample['top_shap_feature'] = top_features
explainability_sample['top_shap_value'] = top_feature_shap
explainability_sample['top_driver_group'] = [map_driver(name) for name in top_features]
explainability_sample['ltv_segment'] = pd.qcut(
    explainability_sample['total_payment_value'].rank(method='first'),
    q=4,
    labels=['ENTRY', 'CORE', 'GROWTH', 'VIP']
)

explainability_sample['recommended_offer_type'] = np.select(
    [
        explainability_sample['top_driver_group'].eq('recency'),
        explainability_sample['top_driver_group'].eq('frequency'),
        explainability_sample['top_driver_group'].eq('logistics'),
        explainability_sample['top_driver_group'].eq('category'),
    ],
    [
        'reactivation_urgency',
        'repeat_purchase_nurturing',
        'logistics_apology_priority_shipping',
        'category_specific_winback',
    ],
    default='value_bundle_offer',
)
explainability_sample['recommended_discount_pct'] = np.select(
    [
        explainability_sample['risk_tier'].eq('HIGH') & explainability_sample['ltv_segment'].eq('VIP'),
        explainability_sample['risk_tier'].eq('HIGH'),
        explainability_sample['risk_tier'].eq('MEDIUM'),
    ],
    [30, 25, 12],
    default=0,
)
explainability_sample['free_shipping_flag'] = np.where(explainability_sample['risk_tier'].isin(['HIGH', 'MEDIUM']), True, False)
explainability_sample['vip_human_touch_flag'] = np.where(
    explainability_sample['risk_tier'].eq('HIGH') & explainability_sample['ltv_segment'].eq('VIP'), True, False
)
explainability_sample.head()


,customer_unique_id,snapshot_key,snapshot_date,recency_days,total_orders,orders_30d,orders_90d,churn_probability,risk_tier,selected_model,version_name,customer_state,avg_review_score,late_delivery_rate_total,revenue_90d,credit_card_share_total,boleto_share_total,voucher_share_total,observed_target,total_payment_value,top_shap_feature,top_shap_value,top_driver_group,ltv_segment,recommended_offer_type,recommended_discount_pct,free_shipping_flag,vip_human_touch_flag
source_row_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,
6225,004288347e5e88a27ded2bb23747066c,20180401,2018-04-01,77,2,0.0000,1.0000,0.9101,LOW,xgboost,v2,RJ,5.0000,0.0000,103.2800,1.0000,0.0000,0.0000,1,354.3700,total_payment_value,0.5775,monetary,GROWTH,value_bundle_offer,0,False,False
6226,00cc12a6d8b578b8ebd21ea4e2ae8b27,20180401,2018-04-01,376,2,0.0000,0.0000,0.9545,LOW,xgboost,v2,SP,4.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1,126.2000,mean_gap_days,0.9899,frequency,ENTRY,repeat_purchase_nurturing,0,False,False
6227,011b4adcd54683b480c4d841250a987f,20180401,2018-04-01,45,2,0.0000,1.0000,0.8569,LOW,xgboost,v2,BA,4.5000,0.0000,149.8800,1.0000,0.0000,0.0000,1,236.3000,recency_days,-0.5439,recency,GROWTH,reactivation_urgency,0,False,False
6228,013f4353d26bb05dc6652f1269458d8d,20180401,2018-04-01,124,2,0.0000,0.0000,0.6468,LOW,xgboost,v2,BA,5.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1,356.3900,mean_gap_days,0.7512,frequency,GROWTH,repeat_purchase_nurturing,0,False,False
6229,015557c9912277312b9073947804a7ba,20180401,2018-04-01,335,2,0.0000,0.0000,0.9673,MEDIUM,xgboost,v2,SP,5.0000,0.0000,0.0000,0.2313,0.7687,0.0000,1,315.1200,recency_days,1.0616,recency,GROWTH,reactivation_urgency,12,True,False


In [6]:
driver_summary = (
    explainability_sample.groupby(['risk_tier', 'top_driver_group', 'recommended_offer_type'], observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        avg_probability=('churn_probability', 'mean'),
        observed_churn_rate=('observed_target', 'mean'),
        avg_discount_pct=('recommended_discount_pct', 'mean'),
    )
    .reset_index()
    .sort_values(['risk_tier', 'rows_n'], ascending=[True, False])
)
driver_summary.to_csv(driver_summary_path, index=False)
explainability_sample.to_parquet(explainability_path, index=False)
logger.info('Explainability parquet saved to %s', explainability_path)
logger.info('Driver summary saved to %s', driver_summary_path)
driver_summary.head(12)

2026-05-06 08:28:33,169 | INFO | Explainability parquet saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_explainability_20260506.parquet


2026-05-06 08:28:33,170 | INFO | Driver summary saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_driver_summary_20260506.csv


,risk_tier,top_driver_group,recommended_offer_type,rows_n,avg_probability,observed_churn_rate,avg_discount_pct
0,HIGH,frequency,repeat_purchase_nurturing,530,0.9930,0.9981,25.7170
2,HIGH,recency,reactivation_urgency,97,0.9917,1.0000,25.2062
1,HIGH,monetary,value_bundle_offer,43,0.9913,1.0000,25.5814
4,LOW,monetary,value_bundle_offer,843,0.7896,0.9537,0.0000
3,LOW,frequency,repeat_purchase_nurturing,380,0.7318,0.9816,0.0000
6,LOW,recency,reactivation_urgency,338,0.8038,0.9704,0.0000
5,LOW,other,value_bundle_offer,112,0.4759,0.8750,0.0000
7,MEDIUM,frequency,repeat_purchase_nurturing,505,0.9766,1.0000,12.0000
8,MEDIUM,monetary,value_bundle_offer,272,0.9721,0.9816,12.0000
10,MEDIUM,recency,reactivation_urgency,217,0.9756,0.9954,12.0000


In [7]:
def figure_to_base64(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', bbox_inches='tight', dpi=160)
    plt.close(fig)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=global_importance.head(12), x='mean_abs_shap', y='feature', ax=axes[0], color='#d62728')
axes[0].set_title('GLOBAL SHAP IMPORTANCE')

plot_driver = (
    explainability_sample['top_driver_group']
    .value_counts(normalize=True)
    .rename_axis('driver')
    .reset_index(name='share')
)
sns.barplot(data=plot_driver, x='share', y='driver', ax=axes[1], color='#1f77b4')
axes[1].set_title('TOP DRIVER MIX IN THE SCORING SAMPLE')
plt.tight_layout()
chart_b64 = figure_to_base64(fig)

html_parts = [
    '<html><head><meta charset="utf-8"><title>Churn Explainability</title></head><body>',
    '<h1>CHURN EXPLAINABILITY REPORT</h1>',
    '<h2>Global SHAP importance</h2>', global_importance.head(25).to_html(index=False),
    '<h2>Driver summary by risk tier</h2>', driver_summary.to_html(index=False),
    '<h2>Sample customer action table</h2>', explainability_sample.head(25).to_html(index=False),
    f'<h2>Visual summary</h2><img src="data:image/png;base64,{chart_b64}" style="max-width:1100px;">',
    '</body></html>'
]
explainability_html_path.write_text('\n'.join(html_parts), encoding='utf-8')
logger.info('Explainability report saved to %s', explainability_html_path)


2026-05-06 08:28:33,546 | INFO | Explainability report saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/churn_explainability_20260506.html


---
## 6.3. NOTEBOOK CLOSURE


The explainability layer now makes the canonical V2C churn model operationally interpretable. Instead of sending one generic campaign to every high-risk customer, VivaMarket can differentiate between inactivity-led churn, logistics frustration, low purchase frequency, or category-specific disengagement.

The next notebook should package that logic into a reusable scoring flow so the same rules can be executed consistently during daily inference.